# 断点位置与标签质量

## tl;dr

1,600 条训练样本，每个距离区间 200 条。单字组 A=110、E=5、U=85；其它区间 U 为 0.5%–4.5%。U 不是已确认错误。AI 逐条初审，尚待人工确认。

## Context & Methods

20 个已固定随机选择的分片，候选 2,475,865 行。来源块 × 距离分层；名额按扩展行数分配；种子 2026100431。

### Key Assumptions

抽样权重与训练损失权重不同。当前原标签可接受不代表唯一答案或全文无噪声。未恢复原文档，无法估计同文档聚类。未加载预测、验证或 test。

In [1]:
from pathlib import Path
import sys, json, io, contextlib, sqlite3, math
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'training/audit_position_quality.py').exists())
sys.path.insert(0, str(root/'training'))
from audit_position_quality import analyze, OUT, REVIEW, sample


## Data

冻结的 sample.jsonl、annotations.jsonl、protocol.json 和 freeze.json 足以复算统计。重新抽样另需原 20 个 ignored 训练分片及 structural.json；可运行 `python3 training/audit_position_quality.py --sample`，脚本会拒绝覆盖不同样本。

In [2]:
with contextlib.redirect_stdout(io.StringIO()):
    result = analyze()
assert all(result['checks'].values())
print(json.dumps(result['checks'],ensure_ascii=False,indent=2))

{
  "frozen_hashes_match": true,
  "all_1600_explicit_annotations": true,
  "all_sample_locations_distinct": true,
  "all_40_sampling_cell_weights_reconcile": true,
  "all_8_strata_have_200": true,
  "no_model_predictions_or_test_loaded": true
}


## Results

### 逐层计数及加权比例

In [3]:
for r in result['absolute']:
    assert r['n']==200 and r['A']+r['E']+r['U']==200
    print(r['group'], {k:r[k] for k in ['n','A','E','U','U_weighted']})

1 {'n': 200, 'A': 110, 'E': 5, 'U': 85, 'U_weighted': 0.4257647838950727}
2 {'n': 200, 'A': 193, 'E': 0, 'U': 7, 'U_weighted': 0.03484845348187905}
3 {'n': 200, 'A': 191, 'E': 0, 'U': 9, 'U_weighted': 0.04500579177112007}
4 {'n': 200, 'A': 194, 'E': 0, 'U': 6, 'U_weighted': 0.029922700853649307}
5–6 {'n': 200, 'A': 197, 'E': 0, 'U': 3, 'U_weighted': 0.015073235100263049}
7–10 {'n': 200, 'A': 196, 'E': 0, 'U': 4, 'U_weighted': 0.0200283553578186}
11–20 {'n': 200, 'A': 199, 'E': 0, 'U': 1, 'U_weighted': 0.004969768100706525}
21+ {'n': 200, 'A': 198, 'E': 0, 'U': 2, 'U_weighted': 0.0100159406459091}


### 从逐条记录独立 SQL 复算

SQLite 对原始 review 表重新分组计算，而不是直接读取汇总表。

In [4]:
query = "SELECT stratum, COUNT(*) n, SUM(verdict='A') A, SUM(verdict='E') E, SUM(verdict='U') U, SUM(weight*(verdict='U'))/SUM(weight) U_weighted FROM review GROUP BY stratum"
with sqlite3.connect(OUT/'report-evidence.sqlite') as con:
    con.row_factory=sqlite3.Row
    sql={r['stratum']:dict(r) for r in con.execute(query)}
for r in result['absolute']:
    assert all(r[k]==sql[r['group']][k] for k in ['n','A','E','U'])
    assert math.isclose(r['U_weighted'], sql[r['group']]['U_weighted'],abs_tol=1e-12)
print('Eight strata: independent SQL counts and weights match.')

Eight strata: independent SQL counts and weights match.


### 相对位置、左右侧及句长敏感性

相对位置跨多个抽样距离层，只读加权比例，不能直接用 U/n 当母体估计。

In [5]:
for r in result['relative']:
    if r['minimum_length']==2:
        print(r['group'],r['n'],r['U_weighted'])
for r in result['sides']:
    if r['stratum']=='1':
        print(r['group'],r['n'],r['U'],r['U_raw'])
for r in result['robustness']:
    if r['scope']=='web; 16–64 chars':
        print(r['scope'],r['group'],r['n'],r['U_weighted'])

0–10% 172 0.04142761478312643
10–20% 290 0.03454703674934977
20–30% 293 0.02272194457589853
30–40% 367 0.025767313011587665
40–50%（中间） 478 0.00417317905962845
left 86 23 0.26744186046511625
right 109 58 0.5321100917431193
symmetric 5 4 0.8
web; 16–64 chars 1 33 0.3938327394081411
web; 16–64 chars 2 84 0.0
web; 16–64 chars 3 61 0.0
web; 16–64 chars 4 66 0.030332787082841275
web; 16–64 chars 5–6 107 0.009337943239362647
web; 16–64 chars 7–10 162 0.024691166017847398
web; 16–64 chars 11–20 165 0.0
web; 16–64 chars 21+ 77 0.012969140902188193


## Takeaways

优先回溯单字尾片段和明确 E，保留正常感叹词、字段、释义等监督。暂不按位置一刀切。独立同分布区间只是抽样参考，不覆盖文档聚类、分片偏差或 AI 判断误差。下一步需原文与人工复核。

In [6]:
assert result == json.loads((REVIEW/'findings.json').read_text())
print('Frozen annotations and report findings reconciled; no model or corpus changed.')

Frozen annotations and report findings reconciled; no model or corpus changed.
